# CS 3110/5110: Data Privacy
## Homework 10

In [103]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1 (10 points)

Implement a function `dp_marginal` that calculates a differentially private one-way marginal for a given column of the adult dataset.

In [104]:
def dp_marginal(col, epsilon):
    hist = adult[col].value_counts()
    noisy_hist = hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    marginal = noisy_hist.clip(lower=0) / noisy_hist.clip(lower=0).sum()
    return marginal




dp_marginal('Occupation', 1.0)

Occupation
Prof-specialty       0.134779
Craft-repair         0.133366
Exec-managerial      0.132374
Adm-clerical         0.122707
Sales                0.118843
Other-service        0.107247
Machine-op-inspct    0.065180
Transport-moving     0.052089
Handlers-cleaners    0.044623
Farming-fishing      0.032360
Tech-support         0.030197
Protective-serv      0.021142
Priv-house-serv      0.004789
Armed-Forces         0.000303
Name: count, dtype: float64

In [105]:
# TEST CASE
marginal = dp_marginal('Age', 1.0)
assert marginal[36] > 0.02 and marginal[36] < 0.03
assert marginal[85] > 0.00005 and marginal[85] < 0.0005

marginal = dp_marginal('Occupation', 1.0)
assert marginal['Prof-specialty'] > 0.13 and marginal['Prof-specialty'] < 0.14
assert marginal['Protective-serv'] > 0.02 and marginal['Protective-serv'] < 0.03

## Question 2 (10 points)

Implement a function `dp_synthetic_data` that generates `n` samples of synthetic data for the given columns, by creating one-way marginals for *each column* and then sampling synthetic data elements for each column separately.

In [106]:
def dp_synthetic_data(cols, n, epsilon):
    def gen_samples(n, marginal):
        return marginal.sample(n=n, replace=True, weights='probability')
    
    syn = pd.DataFrame()

    for col in cols:
        marginal = dp_marginal(col, epsilon)
        
        marginal = marginal.to_frame(name='probability').reset_index()
        marginal.columns = [col, 'probability']

        syn[col] = gen_samples(n, marginal)[col].reset_index(drop=True)
    
    return syn



dp_synthetic_data(['Age', 'Occupation', 'Marital Status', 'Education', 'Relationship'], 100, 1.0)

,Age,Occupation,Marital Status,Education,Relationship
0,60,Adm-clerical,Married-civ-spouse,Some-college,Wife
1,29,Machine-op-inspct,Separated,HS-grad,Husband
2,33,Exec-managerial,Married-civ-spouse,HS-grad,Other-relative
3,60,Exec-managerial,Separated,Bachelors,Unmarried
4,24,Transport-moving,Divorced,HS-grad,Own-child
...,...,...,...,...,...
95,43,Adm-clerical,Widowed,12th,Not-in-family
96,26,Sales,Never-married,Prof-school,Husband
97,20,Farming-fishing,Never-married,Masters,Husband
98,31,Sales,Never-married,Some-college,Husband


In [107]:
# TEST CASE
assert stats.wasserstein_distance(dp_synthetic_data(['Age'], len(adult), 1.0)['Age'], adult['Age']) < 0.2
assert stats.wasserstein_distance(dp_synthetic_data(['Education-Num'], len(adult), 1.0)['Education-Num'], 
                                  adult['Education-Num']) < 0.03

## Question 3 (10 points)

Implement a function `dp_two_marginal` that builds a 2-way marginal with differential privacy.

In [108]:
def dp_two_marginal(col1, col2, epsilon):
    hist = adult[[col1, col2]].value_counts()
    dp_hist = hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon)).clip(lower=0)
    marginal = dp_hist / dp_hist.sum()
    return marginal.to_frame(name='probability').reset_index()


dp_two_marginal('Education', 'Workclass', 1.0)

,Education,Workclass,probability
0,HS-grad,Private,0.253289
1,Some-college,Private,0.165820
2,Bachelors,Private,0.115548
3,Assoc-voc,Private,0.032695
4,11th,Private,0.030051
...,...,...,...
96,5th-6th,Federal-gov,0.000146
97,1st-4th,State-gov,0.000123
98,Preschool,State-gov,0.000022
99,11th,Never-worked,0.000000


In [109]:
# TEST CASE
marginal = dp_two_marginal('Education', 'Workclass', 1.0)
m1 = marginal[(marginal['Education'] == 'HS-grad') & (marginal['Workclass'] == 'Private')]['probability'].values[0]
m2 = marginal[(marginal['Education'] == 'Bachelors') & (marginal['Workclass'] == 'Federal-gov')]['probability'].values[0]
print(m1, m2)
assert m1 > 0.24 and m1 < 0.26
assert m2 > 0.005 and m2 < 0.007

0.253137811679886 0.006970826380572958


## Question 4 (30 points)

Implement a function `dp_synthetic_data_two_marginal` that generates synthetic data for the `Age`, `Workclass`, `Occupation`, and `Education` columns *while preserving correlations between them* by using overlapping 2-way marginals.

take 

Age -> Work -> Occup -> Edu

Age 

Age x Work

Work x Occ

Occ x Edu

not valid actual model but fine for the task



In [110]:
def dp_synthetic_data_two_marginal(n, epsilon):

    def gen_samples(n, marginal):
        return marginal.sample(n=n, replace=True, weights='probability').drop(columns='probability')
    
    def gen_conditional(s,m,cond,target):
        limited = m[m[cond] == s]
        return limited.sample(n=1,weights='probability')[target].iloc[0]

    def dp_marginal(col, epsilon):
        hist = adult[col].value_counts()
        noisy_hist = hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
        marginal = noisy_hist.clip(lower=0) / noisy_hist.clip(lower=0).sum()
        return marginal.to_frame(name='probability').reset_index()

    age_marginal = dp_marginal('Age', epsilon/4.0)
    age_marginal.columns = ['Age', 'probability']

    age_work = dp_two_marginal('Age', 'Workclass', epsilon/4.0)
    work_occ = dp_two_marginal('Workclass', 'Occupation', epsilon/4.0)
    occ_edu = dp_two_marginal('Occupation', 'Education', epsilon/4.0)

    syn = gen_samples(n, age_marginal)
    
    syn['Workclass'] = [gen_conditional(age, age_work, 'Age', 'Workclass') for age in syn['Age']]
    syn['Occupation'] = [gen_conditional(work, work_occ, 'Workclass', 'Occupation') for work in syn['Workclass']]
    syn['Education'] = [gen_conditional(occ, occ_edu, 'Occupation', 'Education') for occ in syn['Occupation']]

    return syn


dp_synthetic_data_two_marginal(100, 1.0)

,Age,Workclass,Occupation,Education
24,45,Private,Sales,HS-grad
38,17,Private,Machine-op-inspct,Some-college
30,51,Federal-gov,Exec-managerial,Bachelors
41,57,Federal-gov,Adm-clerical,Some-college
63,79,Local-gov,Other-service,Some-college
...,...,...,...,...
48,65,Private,Sales,Some-college
48,65,Private,Other-service,10th
37,54,Self-emp-inc,Exec-managerial,Masters
9,25,Private,Other-service,HS-grad


In [111]:
# TEST CASE
synthetic_data = dp_synthetic_data_two_marginal(100, 1.0)

s1 = synthetic_data['Age'].mean()
s2 = len(synthetic_data[synthetic_data['Workclass'] == 'Private'])
s3 = len(synthetic_data[synthetic_data['Occupation'] == 'Adm-clerical'])
s4 = len(synthetic_data[synthetic_data['Education'] == 'Bachelors'])

print(s1, s2, s3, s4)
    
assert s1 > 35 and s1 < 45
assert s2 > 65 and s2 < 90
assert s3 > 5 and s3 < 25
assert s4 > 5 and s4 < 35

38.16 69 15 22
